# 주성분(main_ingredients) 전처리
같은 상품명(`product_name`)이면 다른 날짜 row의 `main_ingredients` 값으로 채움

In [7]:
import pandas as pd
import os

## 1. 데이터 로드 (모든 info CSV 합치기)

In [8]:
DATA_DIR = "[Module]oliveyoung_crawler/Data"

def read_csv_auto(path):
    for enc in ['utf-8-sig', 'cp949', 'utf-8']:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 감지 실패: {path}")

info_files = [
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if 'info' in f and f.endswith('.csv') and not f.startswith('~')
]
print(f"로드할 파일 수: {len(info_files)}")

dfs = [read_csv_auto(f) for f in info_files]
df = pd.concat(dfs, ignore_index=True)
print(f"전체 행 수: {len(df)}")
print(f"main_ingredients 결측: {df['main_ingredients'].isna().sum()} / {len(df)}")

로드할 파일 수: 14
전체 행 수: 336
main_ingredients 결측: 246 / 336


## 2. 상품명 기준 주성분 채우기

In [9]:
# 상품명별로 main_ingredients가 있는 행에서 첫 번째 값을 lookup으로 사용
lookup = (
    df.dropna(subset=['main_ingredients'])
      .drop_duplicates(subset=['product_name'], keep='first')
      .set_index('product_name')['main_ingredients']
)
print(f"lookup 가능한 상품 수: {len(lookup)}")

# 결측 행만 lookup으로 채우기
mask = df['main_ingredients'].isna()
df.loc[mask, 'main_ingredients'] = df.loc[mask, 'product_name'].map(lookup)

print(f"채운 후 결측: {df['main_ingredients'].isna().sum()} / {len(df)}")

lookup 가능한 상품 수: 49
채운 후 결측: 67 / 336


## 3. 결과 확인

In [10]:
# 여전히 결측인 상품 (전체 데이터에 걸쳐 main_ingredients가 한 번도 없는 것)
still_null = df[df['main_ingredients'].isna()]['product_name'].unique()
print(f"채워지지 않은 상품 수: {len(still_null)}")
still_null[:10]

채워지지 않은 상품 수: 33


array(['[변우석포카증정/포켓몬에디션]닥터지 레드블레미쉬 클리어 히알시카 수딩세럼 50ml기획 (리필50ml+꼬부기파우치)',
       '[5/4 하루특가/5월 올영픽]에스테덤 셀룰러워터 에센스 125ml 기획 (+버블 펌프 50ml)',
       '[결광]코스알엑스 더 블루 펩타이드 바쿠치올 플럼프 글로우 세럼 50ml',
       '[기미잡티세럼] 레이어랩 기미흔적 클리어 세럼 30ml',
       '[1+1 리필기획] 이니스프리 레티놀 시카 흔적 앰플 30ml 리필 기획 토이 스토리 5 에디션',
       '[1+1 리필기획] 이니스프리 비타민C 그린티 엔자임 세럼 30ml 리필 기획 토이 스토리 5 에디션',
       '[5월올영픽/포켓몬 에디션] 닥터지 레드블레미쉬 클리어 히알 시카 수딩 세럼 50ml 기획 (리필50ml+꼬부기파우치)',
       '[NEW] 랑콤 레네르지 젤-인-로션 200/400ml',
       '[물광플럼핑]스킨1004 마다가스카르 센텔라 히알루-테카 플럼핑 앰플 50ml',
       '[대용량]스킨1004 마다가스카르 센텔라 앰플 100ml'], dtype=object)

In [11]:
df[['date', 'sort_type', 'product_name', 'main_ingredients']].head(10)

,date,sort_type,product_name,main_ingredients
0,2026-05-01,신상품순,코스알엑스 더 블루 펩타이드 바쿠치올 플럼프 글로우 세럼 50ml,"Peptide(Copper Tripeptide-1, Dipeptide Diamino..."
1,2026-05-01,신상품순,퓨어그램 오리진 부스터샷 300 앰플 30ml,"Spicule(Hydrolyzed Sponge), Ceramide(Ceramide ..."
2,2026-05-01,신상품순,퓨어그램 오리진 부스터샷 100 앰플 30ml,"Spicule(Hydrolyzed Sponge), Ceramide(Ceramide ..."
3,2026-05-01,신상품순,쥬베룩 PDLLA 리파이닝 퍼스트 에센스 150ml,"PDLLA(Polylactic Acid), Peptide(Palmitoyl Trip..."
4,2026-05-01,신상품순,쥬베룩 PDLLA 콜라겐 부스팅 앰플 20g,"PDLLA(Polylactic Acid), Collagen(Hydrolyzed Co..."
5,2026-05-01,신상품순,[리뉴얼/속수분쏙] 리얼베리어 워터리 히알 세럼 50ml 기획 (+20ml),"Hyaluronic Acid(Sodium Hyaluronate), Ceramide(..."
6,2026-05-01,신상품순,[기미/잡티/흔적 세럼] 레이어랩 기미흔적 클리어 세럼 30ml,"Niacinamide(Niacinamide), Tranexamic Acid(Tran..."
7,2026-05-01,신상품순,[토이 스토리 5] 이니스프리 레티놀 시카 흔적 앰플 30ml 리필 기획 토이 스토...,"Retinol(Retinol), Cica(Asiaticoside, Madecassi..."
8,2026-05-01,신상품순,[토이 스토리 5] 이니스프리 비타민C 그린티 엔자임 세럼 30ml 리필 기획 토이...,"Vitamin C(3-O-Ethyl Ascorbic Acid, Ascorbyl Te..."
9,2026-05-01,신상품순,[포켓몬 에디션] VT 피디알엔 에센스 100 30ml 더블 기획 (+포켓몬 장바구...,"PDRN(Sodium DNA, Panax Ginseng Root Extract), ..."


## 4. 파일별로 나눠서 저장 (선택)

In [12]:
OUTPUT_DIR = "[Module]oliveyoung_crawler/Data_processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for src_path in info_files:
    fname = os.path.basename(src_path)
    src_df = read_csv_auto(src_path)

    # 결측인 main_ingredients만 lookup으로 채우기
    mask = src_df['main_ingredients'].isna()
    src_df.loc[mask, 'main_ingredients'] = src_df.loc[mask, 'product_name'].map(lookup)

    out_path = os.path.join(OUTPUT_DIR, fname)
    src_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"저장: {out_path}")

print("완료")

저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_신상품순(info)_260501.csv
저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_신상품순(info)_260502.csv
저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_신상품순(info)_260503.csv
저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_신상품순(info)_260504.csv
저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_신상품순(info)_260505.csv
저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_신상품순(info)_260506.csv
저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_신상품순(info)_260507.csv
저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_판매순(info)_260501.csv
저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_판매순(info)_260502.csv
저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_판매순(info)_260503.csv
저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_판매순(info)_260504.csv


C:\Users\pijun\AppData\Local\Temp\ipykernel_28500\3459055533.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Peptide(Copper Tripeptide-1, Dipeptide Diaminobutyroyl Benzylamide Diacetate, Acetyl Hexapeptide-1, Acetyl Heptapeptide-4, Acetyl Hexapeptide-8, Acetyl Octapeptide-3), Bakuchiol(Bakuchiol), Collagen(Hydrolyzed Collagen), Hyaluronic Acid(Sodium Hyaluronate, Hyaluronic Acid, Sodium Hyaluronate Crosspolymer, Hydrolyzed Hyaluronic Acid, Sodium Acetylated Hyaluronate), Niacinamide(Niacinamide), Adenosine(Adenosine), Tocopherol(Tocopherol)'
 'Spicule(Hydrolyzed Sponge), Ceramide(Ceramide NP, Ceramide AS, Ceramide NS, Ceramide AP, Ceramide EOP), Hyaluronic Acid(Hydrolyzed Hyaluronic Acid, Sodium Hyaluronate, Sodium Hyaluronate Crosspolymer, Potassium Hyaluronate, Hydroxypropyltrimonium Hyaluronate, Hydrolyzed Sodium Hyaluronate, Hyaluronic Acid, Sodium Acetylated Hyaluronate), Amino Acid(Glycine, Glutamic Ac

저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_판매순(info)_260505.csv
저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_판매순(info)_260506.csv
저장: [Module]oliveyoung_crawler/Data_processed\oliveyoung_판매순(info)_260507.csv
완료


C:\Users\pijun\AppData\Local\Temp\ipykernel_28500\3459055533.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Vitamin C(Ascorbyl Glucoside, Ascorbic Acid), Retinol(Retinol, Hydroxypinacolone Retinoate)'
 'Zinc Teca(Asiaticoside, Madecassic Acid, Asiatic Acid, Zinc PCA), TECA(Asiaticoside, Madecassic Acid, Asiatic Acid), Cica(Centella Asiatica Extract, Asiaticoside, Madecassic Acid, Asiatic Acid), Zinc PCA(Zinc PCA), Niacinamide(Niacinamide)'
 nan
 'PDRN(Sodium DNA), Hyaluronic Acid(Hydrolyzed Hyaluronic Acid, Sodium Hyaluronate, Hyaluronic Acid, Hydrolyzed Sodium Hyaluronate, Hydroxypropyltrimonium Hyaluronate, Potassium Hyaluronate, Sodium Hyaluronate Crosspolymer, Sodium Acetylated Hyaluronate), Collagen(Hydrolyzed Collagen)'
 nan nan
 'Niacinamide(Niacinamide), Damask Rose Water(Rosa Damascena Flower Water)'
 'Hyaluronic Acid(Sodium Hyaluronate, Hydrolyzed Hyaluronic Acid, Hydrolyzed Sodium Hyaluronate, So